# YOUTUBE Transcript system Using RAG Pipline

In [64]:
import os
os.environ["RAPID_API_KEY"]="RAPIDAPI_KEY=46994ef205mshb28c0c0d08ea174p18ffb4jsn4d56944b45e5"

In [65]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# Step#1(A): Indexing(Document Ingestion)

[expression  for  variable  in  iterable]
chunk.text   for   chunk    in   fetched


In [66]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

ytt_api = YouTubeTranscriptApi() # this is instance ,which is ab fetch() use kar sakte ho
videoId = "CndSt8d5utY"
#videoId = "rlGrK6dZfCQ"
try:
    fetched = ytt_api.fetch(videoId,languages=['en'])  # get_transcript() nahi, fetch() use karo
    transcript = " ".join(chunk.text for chunk in fetched)  # chunk["text"] nahi, chunk.text
    print(transcript)
except TranscriptsDisabled: 
    print("No caption avail for this video")

Hi, this is Abbe. Welcome to my YouTube channel. In this video, I'll show you how to install Nitan on Oracle Cloud free tier using Docker EngineX and Sbot. By this method, you will have a server free for life as well as you will have less overhead on your server with just only nit installed as well as you will have more control to your server. This method is a text method. You'll need some knowledge of command line interface or Windows uh command line or Mac terminal. But if you want to have a lesser complicated version like graphical method where you can just drag and drop methods or just have a minimal low code environment and get it set up. I have another video made on this topic. You can check the description of this video and uh then you can get started. So let's start. So this is the NIN's website docs.net.io/hosting where you can find all the installation guides. But don't worry, I'm here for you. So I'll be explaining each thing in detail. So the server we'll be using for this 

In [67]:
fetched

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Hi, this is Abbe. Welcome to my YouTube', start=0.16, duration=3.68), FetchedTranscriptSnippet(text="channel. In this video, I'll show you", start=2.24, duration=3.519), FetchedTranscriptSnippet(text='how to install Nitan on Oracle Cloud', start=3.84, duration=5.36), FetchedTranscriptSnippet(text='free tier using Docker EngineX and Sbot.', start=5.759, duration=5.201), FetchedTranscriptSnippet(text='By this method, you will have a server', start=9.2, duration=4.399), FetchedTranscriptSnippet(text='free for life as well as you will have', start=10.96, duration=5.04), FetchedTranscriptSnippet(text='less overhead on your server with just', start=13.599, duration=4.801), FetchedTranscriptSnippet(text='only nit installed as well as you will', start=16.0, duration=4.48), FetchedTranscriptSnippet(text='have more control to your server. This', start=18.4, duration=4.16), FetchedTranscriptSnippet(text="method is a text method. You'll ne

In [68]:
transcript

"Hi, this is Abbe. Welcome to my YouTube channel. In this video, I'll show you how to install Nitan on Oracle Cloud free tier using Docker EngineX and Sbot. By this method, you will have a server free for life as well as you will have less overhead on your server with just only nit installed as well as you will have more control to your server. This method is a text method. You'll need some knowledge of command line interface or Windows uh command line or Mac terminal. But if you want to have a lesser complicated version like graphical method where you can just drag and drop methods or just have a minimal low code environment and get it set up. I have another video made on this topic. You can check the description of this video and uh then you can get started. So let's start. So this is the NIN's website docs.net.io/hosting where you can find all the installation guides. But don't worry, I'm here for you. So I'll be explaining each thing in detail. So the server we'll be using for this

In [69]:
for chunk in fetched:
    print(f"Start: {chunk.start:.1f}s | Duration: {chunk.duration:.1f}s | Text: {chunk.text}")

Start: 0.2s | Duration: 3.7s | Text: Hi, this is Abbe. Welcome to my YouTube
Start: 2.2s | Duration: 3.5s | Text: channel. In this video, I'll show you
Start: 3.8s | Duration: 5.4s | Text: how to install Nitan on Oracle Cloud
Start: 5.8s | Duration: 5.2s | Text: free tier using Docker EngineX and Sbot.
Start: 9.2s | Duration: 4.4s | Text: By this method, you will have a server
Start: 11.0s | Duration: 5.0s | Text: free for life as well as you will have
Start: 13.6s | Duration: 4.8s | Text: less overhead on your server with just
Start: 16.0s | Duration: 4.5s | Text: only nit installed as well as you will
Start: 18.4s | Duration: 4.2s | Text: have more control to your server. This
Start: 20.5s | Duration: 3.9s | Text: method is a text method. You'll need
Start: 22.6s | Duration: 4.5s | Text: some knowledge of command line interface
Start: 24.4s | Duration: 5.8s | Text: or Windows uh command line or Mac
Start: 27.0s | Duration: 5.0s | Text: terminal. But if you want to have a
Start: 30.2s

# Step1(B) Indexing (Text Splitting)

RecursiveCharacterTextSplitter kya karta hai
Lambi string ko chhote meaningful pieces mein kaatta hai — smartly
"Recursive" isliye kehte hain kyunke yeh pehle \n\n pe kaatne ki koshish karta hai, phir \n pe, phir . pe, phir   pe — jab tak chunk size fit na ho jaye
Priority:  "\n\n"  →  "\n"  →  "."  →  " "


transcript (1 lambi string)
         ↓
create_documents([transcript])
         ↓
[
  Document("In this video...API key"),      # chunk 1 — 500 chars
  Document("...API key from GCP console"),  # chunk 2 — 500 chars (100 overlap)
  Document("...GCP console mein jaake"),    # chunk 3 — 500 chars (100 overlap)
  ...
]

In [70]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100 # # 100 characters dono chunks mein common honge
)
chunks = splitter.create_documents([transcript]) # Yeh method plain string ko LangChain Document objects mein convert karta hai:

In [71]:
len(chunks)

48

In [72]:
chunks[0]

Document(metadata={}, page_content="Hi, this is Abbe. Welcome to my YouTube channel. In this video, I'll show you how to install Nitan on Oracle Cloud free tier using Docker EngineX and Sbot. By this method, you will have a server free for life as well as you will have less overhead on your server with just only nit installed as well as you will have more control to your server. This method is a text method. You'll need some knowledge of command line interface or Windows uh command line or Mac terminal. But if you want to have a")

# Step#1(C) and 1(D) = Indexing (Embedding Generation and storing in Vector Store)

In [73]:
from dotenv import load_dotenv
load_dotenv(r"E:\AI_ML_Python\Deeplearning\Langchain_Model\.env")
embedding = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2', # ismian numbers ki length 384 hogi son yeh har chunk amin vector 384 numbers rakhta hai
)
vector_store = FAISS.from_documents(chunks,embedding)   

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6377.90it/s]


In [74]:
embedding

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [75]:
vector_store.index_to_docstore_id


{0: '16598bef-ffd9-4da0-9cfd-38abbde693e7',
 1: 'a53f8f03-0ae6-47b3-ac06-f90661650406',
 2: '5e542403-57da-436f-9239-369d7afed7d4',
 3: '7ae28671-8e8f-4ee0-b12d-b78f5950b578',
 4: 'b71af3c1-9a02-462a-89be-5cf160b7f513',
 5: '97a0c34f-b17f-43a7-a3d5-d42822e1c7ae',
 6: 'd67e34f2-d08c-4352-814a-c344cb76dc77',
 7: 'e6342fcc-ecc2-41e8-b100-2904d214033a',
 8: 'edb6a29f-6026-4afc-a908-4ba6d304d4e4',
 9: '275a342f-074d-4cf3-a289-9868c2ff3bc1',
 10: '4cbff67d-b5b3-4f94-aadf-b89de8e935fa',
 11: 'ce980496-a6d8-405b-93f1-542da04f4f69',
 12: 'ec948a59-2fa1-4c4d-9fb6-7017e7301e43',
 13: '867b26ef-a4a4-475e-a768-94cc00a61e21',
 14: 'b5ab2bb9-33cd-420d-b951-d5263921b120',
 15: '4fb0ec56-bf94-4e0c-bc1b-9307be4f7638',
 16: 'c3719fea-d10b-48de-8e53-b4c853e9f4e5',
 17: '265ee0c6-a59c-4821-8b0e-f463016036fa',
 18: '26fb583e-0f10-4c0f-a766-c0c326a7ea45',
 19: 'a8df9ce0-db61-4ee2-a679-90f2002791e0',
 20: 'a519ca50-78c3-425a-ad04-f52924202fee',
 21: '810fd318-5991-4cbb-9b7b-698f7bb5a9d8',
 22: '613b494e-1d52-

In [76]:
vector = vector_store.index.reconstruct(0)
print(vector)
print(vector.shape)



[-5.51902503e-02  1.63661446e-02 -1.16849216e-02 -1.13270931e-01
  2.16191243e-02 -3.34082395e-02 -8.37241858e-03  2.22870577e-02
 -9.42349061e-03  4.27977145e-02  1.01634162e-02 -1.13210551e-01
 -1.53546333e-02 -5.83597012e-02  5.61965182e-02 -1.64527595e-02
  9.87108983e-03 -9.59318131e-03  7.74197504e-02 -3.73699181e-02
 -5.75523302e-02  2.73672007e-02 -5.43457232e-02 -7.43590891e-02
 -3.72438878e-02  9.44510754e-03  5.57191204e-03 -5.76230884e-02
  4.91874292e-02 -4.22092751e-02 -2.75539011e-02 -6.75787171e-03
 -1.37334066e-02 -1.71619151e-02 -2.39634532e-02  2.65113935e-02
 -4.27959068e-03  2.50421781e-02 -1.32856503e-01 -2.00309772e-02
  1.38055086e-02 -9.01564360e-02 -1.31968617e-01 -7.21485587e-03
  3.24570909e-02 -1.77421123e-02 -1.48024829e-02 -8.59422609e-02
  6.56444281e-02 -1.60781052e-02 -7.39832595e-02 -4.25338112e-02
  1.48996906e-02  2.43937578e-02  7.72632733e-02 -1.00830324e-01
 -4.82703373e-02  1.91767551e-02  1.17359646e-02  1.54537996e-02
  3.95874754e-02 -6.62764

In [77]:
for i in range(vector_store.index.ntotal):
    vector = vector_store.index.reconstruct(i)
    rounded = [round(float(x), 47) for x in vector[:3]]  # sirf pehle 3 numbers
    print(f"Chunk {i}: {rounded} ... ← 384 numbers")

Chunk 0: [-0.055190250277519226, 0.01636614464223385, -0.011684921570122242] ... ← 384 numbers
Chunk 1: [0.006978094112128019, -0.07502918690443039, -0.01946539618074894] ... ← 384 numbers
Chunk 2: [0.0019445883808657527, -0.05706552788615227, 0.022764582186937332] ... ← 384 numbers
Chunk 3: [-0.016983816400170326, 0.0012385912705212831, -0.008039554581046104] ... ← 384 numbers
Chunk 4: [0.03724569454789162, -0.04880274087190628, -0.006722770631313324] ... ← 384 numbers
Chunk 5: [-0.028677923604846, -0.03683288395404816, -0.019795462489128113] ... ← 384 numbers
Chunk 6: [-0.04039767012000084, -0.07567060738801956, 0.012925463728606701] ... ← 384 numbers
Chunk 7: [-0.017419204115867615, -0.04085496440529823, -0.01826220192015171] ... ← 384 numbers
Chunk 8: [0.05575202405452728, -0.003236769000068307, 0.035440538078546524] ... ← 384 numbers
Chunk 9: [0.07972218096256256, 0.011396991088986397, -0.04556116461753845] ... ← 384 numbers
Chunk 10: [0.03882906213402748, -0.028114762157201767, -

In [78]:
vector_store.get_by_ids(['56ef173a-3e50-47f5-9fae-c1a9fd6e4727'])

[]

In [79]:
for i in range(vector_store.index.ntotal):
    doc_id = vector_store.index_to_docstore_id[i]
    doc = vector_store.docstore.search(doc_id) # Docstore = ek dictionary hai jo UUID se text store karta hai .search(doc_id) = us dictionary mein se UUID de do, Document wapas lo:
    print(f"Chunk {i}: {doc.page_content[:100]}...")
    print("---")

Chunk 0: Hi, this is Abbe. Welcome to my YouTube channel. In this video, I'll show you how to install Nitan o...
---
Chunk 1: of command line interface or Windows uh command line or Mac terminal. But if you want to have a less...
---
Chunk 2: NIN's website docs.net.io/hosting where you can find all the installation guides. But don't worry, I...
---
Chunk 3: so it's /in. So you can uh go to their website and understand how oracle free works. So we need comp...
---
Chunk 4: 1 GB memory and as well as one core CPU each as well as with this you will have one ARM compute inst...
---
Chunk 5: run multiple workflows in nutan. So let's start how you can get started with it. So uh to visit you ...
---
Chunk 6: You'll have to add it and verify your account and get started with the free tier. Once you sign up w...
---
Chunk 7: create a option named as create a VM instance on the build screen. So I'll select this option and cl...
---
Chunk 8: availability domain. Then clicking on the image and sha

# Step#2 : Retrieval

In [80]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})


In [81]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019E838EFE10>, search_kwargs={'k': 4})

In [82]:
retriever.invoke("what is youtube api")

[Document(id='613b494e-1d52-405c-94c0-116446588a1d', metadata={}, page_content="private key. So, I'll give you a few commands by which you can uh protect this key. I have this command pasted in the description of the video. Uh this is the permissions. You will have to make the permissions to 600. Ch 600 space the key path of the key where the key is stored. So, this is only for Mac. Make sure that if you're using Windows, you can follow the uh video available in the description and move to the next time stamp available. So, um I'll just clear this interface and I'll just"),
 Document(id='c3a8d0f1-da80-4323-ab23-bae6bacc1e3d', metadata={}, page_content='this video. If you have if this video has been helpful for you, you can drop me in comments. If you have any suggestions, please let me know in comments through. If you need any of my assistance in setting up this edit, you can reach out to via my email address or social channels. Links are available in the description. Have a great day.

# Step#3 Augmentation

In [83]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

In [84]:
prompt = PromptTemplate(
    template="""
                you are helpful assistant.
                Answer only from the provided transcript context
                if the context is insufficient just say you don't knwon
                {context}
                Question:{question} 
""",
input_variables=['context','question']
)

In [85]:
questions = "is the topic of n8n discussed in this viedo? if yes then what was discussed"
retrieved_docs = retriever.invoke(questions)
retrieved_docs

[Document(id='69cb4e20-6bf7-4a24-bc1b-5e9cb5334571', metadata={}, page_content="means the n is actually working on your server. So the next step is that uh you have to check if the n is successfully installed on your server or not. for which you'll have to go to your uh Oracle cloud again and you have to copy this public IP address. Once copied you'll have to go to your browser and just type col 5678. So by this method you can see that uh your server is configured to use a secure cookie URL. That means the setup has been completed. However uh it is not secure here. So only"),
 Document(id='c3719fea-d10b-48de-8e53-b4c853e9f4e5', metadata={}, page_content="the that's the um basic um HTTP and HTTPS uh ports here. So you have to just enter the comma and you'll be able to add these ports here and click on add ingress rules. Once these rules are added so you can communicate with with your servers via HTTP as well as HTTPS protocol. Then you'll have to click on again add ingress rule. You don

In [86]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text


"means the n is actually working on your server. So the next step is that uh you have to check if the n is successfully installed on your server or not. for which you'll have to go to your uh Oracle cloud again and you have to copy this public IP address. Once copied you'll have to go to your browser and just type col 5678. So by this method you can see that uh your server is configured to use a secure cookie URL. That means the setup has been completed. However uh it is not secure here. So only\n\nthe that's the um basic um HTTP and HTTPS uh ports here. So you have to just enter the comma and you'll be able to add these ports here and click on add ingress rules. Once these rules are added so you can communicate with with your servers via HTTP as well as HTTPS protocol. Then you'll have to click on again add ingress rule. You don't have to change anything. Type 0.0.0/ here and type 5678 because that's the port of the N10 that will be installing on our server. So you'll have to make sur

In [87]:
final_prompt = prompt.invoke({"context":context_text, "question":questions})
final_prompt

StringPromptValue(text="\n                you are helpful assistant.\n                Answer only from the provided transcript context\n                if the context is insufficient just say you don't knwon\n                means the n is actually working on your server. So the next step is that uh you have to check if the n is successfully installed on your server or not. for which you'll have to go to your uh Oracle cloud again and you have to copy this public IP address. Once copied you'll have to go to your browser and just type col 5678. So by this method you can see that uh your server is configured to use a secure cookie URL. That means the setup has been completed. However uh it is not secure here. So only\n\nthe that's the um basic um HTTP and HTTPS uh ports here. So you have to just enter the comma and you'll be able to add these ports here and click on add ingress rules. Once these rules are added so you can communicate with with your servers via HTTP as well as HTTPS proto

# Step#4 Generations

In [88]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of n8n is discussed in this video. 

The discussion about n8n includes the following points:

1. Checking if the n is successfully installed on the server or not.
2. Configuring the server to use a secure cookie URL.
3. Adding ingress rules for HTTP and HTTPS protocols.
4. Installing n8n on the server and checking if it's working.
5. Referencing n8n's website for installation guides.


# Above have manually work of each work separatly
# Building Chain =  so in this we have jsut call one time invoke to call runaable and so on to automatote

YTT RAG PIPLINE
Video ID 
   ↓
Transcript (raw text)          ← YouTubeTranscriptApi
   ↓
Chunks (500 char pieces)       ← RecursiveCharacterTextSplitter  
   ↓
Vectors (384 numbers)          ← HuggingFaceEmbeddings
   ↓
FAISS Index (stored vectors)   ← FAISS.from_documents()
   ↓
Question → Top 4 chunks        ← retriever.invoke()
   ↓
Prompt = Context + Question    ← PromptTemplate
   ↓
Answer                         ← ChatGroq (Llama)

In [89]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
# RunnableLambda  = simple lambda function ko runnable chain mein wrap karta hai
# RunnableParallel = multiple chains ek saath parallel chalata hai (yahan use nahi hua)
from langchain_core.output_parsers import StrOutputParser
# LLM ka raw output clean karta hai

In [90]:
def format_doc(retrieved_docs):
    context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [91]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_doc),
    'question': RunnablePassthrough()
})

In [92]:
parallel_chain.invoke("what is Youtube Api?")

{'context': "private key. So, I'll give you a few commands by which you can uh protect this key. I have this command pasted in the description of the video. Uh this is the permissions. You will have to make the permissions to 600. Ch 600 space the key path of the key where the key is stored. So, this is only for Mac. Make sure that if you're using Windows, you can follow the uh video available in the description and move to the next time stamp available. So, um I'll just clear this interface and I'll just\n\nthis video. If you have if this video has been helpful for you, you can drop me in comments. If you have any suggestions, please let me know in comments through. If you need any of my assistance in setting up this edit, you can reach out to via my email address or social channels. Links are available in the description. Have a great day. Bye. [Music]\n\nmeans the n is actually working on your server. So the next step is that uh you have to check if the n is successfully installed o

In [93]:
parser = StrOutputParser()

In [94]:
main_chain = parallel_chain | prompt | llm | parser

In [95]:
main_chain.invoke("can you summarize the video")

'The video is about protecting a private key and setting up a server. The speaker provides commands to change permissions to 600 for the key on a Mac. They also mention a public IP address and connecting to the server using a notepad on a Mac or Putty on Windows. Additionally, the video discusses setting up a free server with 1 GB memory, one core CPU, and 3,000 OCP hours a month.'